# T.E.R.R.A. — Stage 2 seismicity case-study starter

**Purpose:** Line up Emily’s forecast windows with an independent earthquake catalog and, *only if available*, measured forecast errors. **Not an eruption predictor.**

**Tonight:** Run sections 1–3; obtain the forecast dates, choose a volcano, and test the catalog workflow. **Tomorrow:** Upload Emily’s `matched_case_metrics.csv` and run section 4.

Data provenance: forecast dates from the T.E.R.R.A. Scientific Demo; earthquake events requested from the [USGS FDSN event API](https://earthquake.usgs.gov/fdsnws/event/1/). **Global USGS catalogs may omit small volcanic earthquakes.** Local observatory catalogs are preferable for detailed unrest analysis.

⚠️ Your 68 rows include multiple forecast horizons from the same origin. They are **not 68 independent events**. No causal claims or operational warnings should be made from exploratory plots.

## 1 · Imports and forecast-date CSV
Export using the snippet in the chat so `case_index` is preserved. This notebook can also show an *illustrative date-only example* if the CSV is unavailable.

In [ ]:
import io, os, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set True in Colab to upload your full forecast-date CSV.
UPLOAD_DATES = True

if UPLOAD_DATES:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Upload terra_forecast_dates.csv")
    filename = next(iter(uploaded))
    dates = pd.read_csv(io.BytesIO(uploaded[filename]))
    example_only = False
else:
    # Only reproduces two date examples from the user's shared table.
    # This is NOT the full 68-row scientific dataset.
    example_only = True
    dates = pd.DataFrame([
       ['corbetti','corbetti_079D_08294_131313','2023-06-02','2023-06-14',12],
       ['tambora','tambora_134D_09864_100610','2023-06-17','2023-06-29',12],
    ], columns=['volcano','source_id','forecast_origin','target_date','horizon_days'])

required = ['volcano','source_id','forecast_origin','target_date','horizon_days']
missing = set(required)-set(dates.columns)
if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
for c in ['forecast_origin','target_date']:
    dates[c] = pd.to_datetime(dates[c], utc=True)
dates['horizon_days'] = pd.to_numeric(dates['horizon_days'])
actual_horizon = (dates.target_date-dates.forecast_origin).dt.total_seconds()/86400
if not np.allclose(actual_horizon, dates.horizon_days, atol=0.01):
    raise ValueError('At least one horizon does not match origin/target dates.')
dates = dates.sort_values(['volcano','forecast_origin','target_date']).reset_index(drop=True)
print('Rows:',len(dates),'Volcanoes:',sorted(dates.volcano.unique()))
if example_only: print('EXAMPLE DATES ONLY — not the full team data.')
display(dates.head(8))

## 2 · Select one volcano and define the catalog search
For this starter we include *verified* coordinates for Etna, Mayon and Merapi. You can add a different volcano only after confirming its coordinates from the Smithsonian Global Volcanism Program. Do not substitute a random coordinate.

In [ ]:
VOLCANO_COORDS = {
    'etna': (37.748, 14.999),
    'mayon': (13.257, 123.685),
    'merapi': (-7.540, 110.446),
}

available = [v for v in dates.volcano.unique() if v in VOLCANO_COORDS]
print('Supported volcanoes present:',available)
if not available:
    raise ValueError('None of the supported volcanoes are in the uploaded CSV. Add verified lat/lon to VOLCANO_COORDS.')

VOLCANO = 'etna' if 'etna' in available else available[0]
RADIUS_KM = 40      # exploratory radius, NOT an established volcano-specific monitoring region
MIN_MAGNITUDE = 2.0 # USGS global catalog may miss smaller local volcano-tectonic events

cases = dates.loc[dates.volcano == VOLCANO].copy()
start = cases.forecast_origin.min() - pd.Timedelta(days=30)
end = cases.target_date.max() + pd.Timedelta(days=1)
print(f'{VOLCANO}: {len(cases)} forecast pairs from {start.date()} to {end.date()}')
print('Distinct forecast origins:',cases.forecast_origin.nunique())
display(cases[['forecast_origin','target_date','horizon_days']].head(12))

## 3 · Independent earthquake catalog
Uses the documented USGS FDSN API. If the API times out or returns no small earthquakes, do not interpret that as no unrest. A local volcano observatory may record events not included in the USGS global feed. **No synthetic earthquake observations are generated here.**

In [ ]:
# Use True when internet is available in Google Colab.
FETCH_USGS = True

def get_usgs_earthquakes(lat, lon, start, end, radius_km=40, min_magnitude=2.0):
    params = dict(format='geojson', starttime=start.strftime('%Y-%m-%d'),
                  endtime=end.strftime('%Y-%m-%d'), latitude=lat,
                  longitude=lon, maxradiuskm=radius_km,
                  minmagnitude=min_magnitude, limit=20000)
    response = requests.get('https://earthquake.usgs.gov/fdsnws/event/1/query',
                            params=params, timeout=60)
    response.raise_for_status()
    data=response.json()
    if data.get('metadata',{}).get('count',0)>=20000:
        raise RuntimeError('Catalog query hit the result limit; split the time range.')
    rows=[]
    for event in data.get('features',[]):
        p=event['properties']; coords=event['geometry']['coordinates']
        rows.append(dict(event_id=event['id'],time=pd.to_datetime(p['time'],unit='ms',utc=True),
                         magnitude=p['mag'],depth_km=coords[2],latitude=coords[1],longitude=coords[0],
                         source='USGS FDSN'))
    columns=['event_id','time','magnitude','depth_km','latitude','longitude','source']
    return pd.DataFrame(rows,columns=columns), response.url

if FETCH_USGS:
    try:
        quakes, request_url = get_usgs_earthquakes(*VOLCANO_COORDS[VOLCANO],start,end,
                                                   RADIUS_KM,MIN_MAGNITUDE)
        print('Fetched',len(quakes),'earthquakes from:',request_url)
    except Exception as e:
        print('No catalog downloaded:',type(e).__name__,str(e))
        quakes=pd.DataFrame(columns=['event_id','time','magnitude','depth_km','latitude','longitude','source'])
else:
    quakes=pd.DataFrame(columns=['event_id','time','magnitude','depth_km','latitude','longitude','source'])
    print('Catalog fetching disabled. No earthquakes assumed.')

# Aggregate seismicity into the same *calendar days* used by each forecast pair.
# [origin, target) is retrospective activity DURING the forecast window; it was NOT available at origin.
def count_during_windows(frame, catalog):
    out = frame.copy()
    if len(catalog):
        t=pd.to_datetime(catalog['time'],utc=True)
        out['earthquakes_during_window']=[int(((t>=a)&(t<b)).sum())
                                         for a,b in zip(out.forecast_origin,out.target_date)]
        out['events_per_day'] = out.earthquakes_during_window/out.horizon_days
    else:
        out['earthquakes_during_window'] = np.nan
        out['events_per_day'] = np.nan
    return out

windowed = count_during_windows(cases,quakes)
display(windowed[['forecast_origin','target_date','horizon_days','earthquakes_during_window','events_per_day']].head(12))

fig,ax=plt.subplots(figsize=(10,3.5))
if len(quakes):
    counts=quakes.set_index('time').resample('7D').size()
    ax.plot(counts.index,counts.values,marker='o')
    ax.set_ylabel('USGS events per 7 days')
else:
    ax.text(.5,.5,'No events available in selected catalog / query',ha='center',va='center',transform=ax.transAxes)
for t in cases.forecast_origin.drop_duplicates().sort_values():
    ax.axvline(t,color='gray',alpha=.17)
ax.set_title(f'{VOLCANO}: seismicity and forecast origins (gray lines)')
ax.set_xlabel('Date (UTC)')
plt.tight_layout();plt.show()

## 4 · Tomorrow: bring in Emily’s forecast errors

Upload `matched_case_metrics.csv` produced by `demo.export()`. **Best option:** Re-export the dates with `case_index` so the merge is exact (snippet in the chat). Never join on volcano + horizon alone; those repeat for different forecast origins.

This cell will refuse an ambiguous merge rather than silently produce false correlations.

In [ ]:
UPLOAD_METRICS = False  # Change to True when you have matched_case_metrics.csv

if UPLOAD_METRICS:
    from google.colab import files
    uploads=files.upload()
    if not uploads: raise ValueError('Upload matched_case_metrics.csv')
    metrics=pd.read_csv(io.BytesIO(next(iter(uploads.values()))))
    if 'case_index' not in dates.columns or 'case_index' not in metrics.columns:
        raise ValueError('case_index missing. Re-export forecast-date CSV with case_index; see chat instructions.')
    # The original subset in `cases` still contains case_index from dates.
    cols=['case_index','model','seed','MAE_mm','extreme_MAE_mm','coverage90','bias_mm']
    cols=[c for c in cols if c in metrics.columns]
    merged=cases.merge(metrics[cols],on='case_index',how='inner',validate='one_to_many')
    if merged.empty: raise ValueError('No matching case_index values.')
    merged = count_during_windows(merged,quakes)
    display(merged.head())
    print('Merged rows:',len(merged),'Unique forecast cases:',merged.case_index.nunique())
    if merged['events_per_day'].notna().any():
        for model in ['persistence','EMA30','lowrank','CVAE']:
            sub=merged[merged.model==model]
            if len(sub):
                fig,ax=plt.subplots(figsize=(5,3.5))
                ax.scatter(sub.events_per_day,sub.MAE_mm,alpha=.6)
                ax.set(xlabel='USGS earthquakes per day during forecast window',
                       ylabel='Retrospective forecast MAE (mm)',
                       title=f'{VOLCANO}: {model} exploratory comparison')
                plt.tight_layout();plt.show()
    merged.to_csv('terra_stage2_joined_cases.csv',index=False)
else:
    print("Waiting for Emily's forecast-error CSV. Sections 1–3 work independently.")

## 5 · Export your reproducible starting evidence

The catalog and window summaries help your team check availability and time alignment. The catalog is a **global earthquake listing**, not a complete volcano observatory seismicity record. `earthquakes_during_window` is descriptive and uses observations *after* forecast origin, so **it is never a legitimate predictive input at forecast origin**.

For an independent scientific test, choose a volcano-specific seismic catalog and obtain a truly untouched chronological holdout. A correlation from a few overlapping retrospective windows is not an eruption forecast.

In [ ]:
from pathlib import Path
out=Path('terra_stage2_results');out.mkdir(exist_ok=True)
dates.to_csv(out/'forecast_dates_checked.csv',index=False)
windowed.to_csv(out/f'{VOLCANO}_forecast_windows.csv',index=False)
quakes.to_csv(out/f'{VOLCANO}_usgs_events.csv',index=False)
print('Saved:',*[str(x) for x in out.iterdir()],sep='\n  ')
